# Superstore Sales & Profitability Analysis

**Portfolio project | Data Analyst Intern**  
**Công cụ:** Google Colab, Python, Pandas, Matplotlib, Seaborn  
**Nguồn dữ liệu:** [Sample - Superstore Dataset (Kaggle)](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)

> **Quy tắc:** Mọi KPI, nhận xét và đề xuất có chứa số liệu đều được tạo **sau khi tải CSV thật**. Notebook không chứa dữ liệu mô phỏng hoặc thành tích giả định. Dữ liệu Superstore là bộ dữ liệu minh họa bán lẻ, không phải kết quả kinh doanh của doanh nghiệp thật.

## 1. Mục tiêu và câu hỏi kinh doanh

**Bài toán:** Đánh giá hiệu quả doanh số và lợi nhuận của hoạt động bán lẻ, phát hiện khu vực/nhóm hàng có biên lợi nhuận thấp, và kiểm tra mối liên hệ giữa giảm giá và lợi nhuận.

1. Doanh số và lợi nhuận thay đổi theo thời gian ra sao?
2. `Category` và `Sub-Category` nào có hiệu quả tốt hoặc bị lỗ?
3. `Region` và `Segment` nào có Profit Margin cao/thấp?
4. Sản phẩm nào có doanh số cao nhưng biên lợi nhuận thấp hoặc âm?
5. `Discount` có **tương quan** như thế nào với `Profit` (không suy diễn quan hệ nhân quả)?

**Đơn vị phân tích:** Mỗi dòng CSV thường là một **dòng hàng** trong đơn. `Order ID` có thể lặp lại; số đơn hàng phải tính bằng số `Order ID` duy nhất.

**KPI:** `Total Sales = SUM(Sales)`; `Total Profit = SUM(Profit)`; `Profit Margin = Total Profit / Total Sales × 100%`; `Total Orders = DISTINCTCOUNT(Order ID)`; `AOV = Total Sales / Total Orders`.

**Quy trình:** Tải dữ liệu → Khám phá → Kiểm tra chất lượng/làm sạch → KPI → Phân tích & trực quan hóa → Insight có bằng chứng → Đề xuất có điều kiện.

## 2. Chuẩn bị môi trường & tải dữ liệu

Chạy cell đầu tiên, bấm **Choose Files**, chọn CSV đã tải từ Kaggle. Không dùng Kaggle API, không cần cài thêm thư viện trên Colab tiêu chuẩn. Nếu tải nhiều CSV cùng lúc, notebook sẽ yêu cầu chọn lại một file để tránh phân tích nhầm.

In [ ]:
from io import BytesIO
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from IPython.display import display, Markdown
from google.colab import files

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (10, 5), "axes.titlesize": 13})
OUTPUT_DIR = Path("superstore_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

uploaded = files.upload()
csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_files:
    raise ValueError("Chưa tải lên file CSV. Hãy chạy lại cell và chọn file .csv.")
preferred = [name for name in csv_files if Path(name).name.lower() == "sample - superstore.csv"]
if preferred:
    file_name = preferred[0]
elif len(csv_files) == 1:
    file_name = csv_files[0]
else:
    raise ValueError(f"Có nhiều file CSV: {csv_files}. Hãy chỉ tải lên 1 file Superstore.")

for enc in ("utf-8-sig", "cp1252"):
    try:
        raw = pd.read_csv(BytesIO(uploaded[file_name]), encoding=enc,
                          dtype={"Postal Code": "string"}, low_memory=False)
        break
    except UnicodeDecodeError:
        continue
else:
    raise ValueError("Không đọc được mã hóa CSV. Kiểm tra lại nguồn file.")

raw.columns = raw.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip()
print(f"File: {file_name} | Encoding: {enc} | Kích thước: {raw.shape[0]:,} dòng × {raw.shape[1]} cột")
display(raw.head())

## 3. Khám phá dữ liệu (Data Understanding)

Kiểm tra kích thước, tên cột, kiểu dữ liệu, giá trị thiếu, dòng trùng và thống kê mô tả **trước** khi làm sạch. Không coi `Order ID` bị lặp là lỗi, vì một đơn hàng có thể có nhiều dòng sản phẩm.

In [ ]:
required = ["Order ID", "Order Date", "Sales", "Profit", "Discount",
            "Category", "Sub-Category", "Region", "Segment", "Product Name"]
missing_columns = [c for c in required if c not in raw.columns]
if missing_columns:
    raise ValueError(
        "CSV không có các cột bắt buộc: " + ", ".join(missing_columns)
        + ". Hãy kiểm tra đúng phiên bản Sample - Superstore."
    )

print(f"Số dòng: {len(raw):,} | Số cột: {raw.shape[1]}")
print("\nKiểu dữ liệu:")
raw.info()
print("\nDanh sách cột:", list(raw.columns))

missing_audit = pd.DataFrame({
    "Missing count": raw.isna().sum(),
    "Missing (%)": (raw.isna().mean() * 100).round(2)
}).sort_values("Missing count", ascending=False)
print("\nGiá trị thiếu theo cột:")
display(missing_audit)
print(f"Số dòng trùng hoàn toàn: {raw.duplicated().sum():,}")
if "Row ID" in raw.columns:
    print(f"Số Row ID bị lặp: {raw['Row ID'].duplicated().sum():,}")
print(f"Số Order ID duy nhất (thô): {raw['Order ID'].nunique(dropna=True):,}")
print("\nThống kê mô tả:")
display(raw.describe(include="all").T)

## 4. Làm sạch dữ liệu và kiểm tra tính hợp lệ

**Nguyên tắc xử lý**

- Giữ `raw` nguyên vẹn để đối chiếu. Chỉ xóa **dòng trùng hoàn toàn** khi có `Row ID` để nhận diện dòng hàng; nếu `Row ID` bị xung đột, dừng để kiểm tra thủ công.
- Chuyển `Sales`, `Profit`, `Discount` về số; `Order Date`, `Ship Date` về kiểu ngày theo định dạng **tháng/ngày/năm** của Superstore. Hỗ trợ thêm ngày ISO (`YYYY-MM-DD`). Không đoán ngày/tháng từ chuỗi mơ hồ.
- Bỏ dòng không có `Order ID`, `Sales` hoặc `Profit` hợp lệ: không thể dùng chúng cho bộ KPI nhất quán. In số lượng đã loại.
- Giữ `Profit` âm: đó có thể là đơn/lô hàng lỗ, **không phải lỗi nhập liệu mặc định**. Không tự xóa `Sales <= 0`; chỉ cảnh báo để rà soát.
- Dòng có ngày không hợp lệ vẫn được tính vào KPI tổng; chỉ không tham gia phân tích xu hướng thời gian. `Discount` ngoài `[0,1]` chỉ bị loại khỏi **phân tích Discount**, không tự thay đổi doanh số/lợi nhuận.
- Giá trị thiếu ở chiều phân tích được gắn `Unknown`, không tự gán vào khu vực/phân khúc khác.

In [ ]:
df = raw.copy()
before_rows = len(df)

# Chuẩn hóa cột dạng text, tránh ghép nhầm nhóm do khoảng trắng.
text_cols = ["Order ID", "Category", "Sub-Category", "Region",
             "Segment", "Product Name", "Product ID"]
for col in text_cols:
    if col in df.columns:
        df[col] = df[col].astype("string").str.strip()
        df[col] = df[col].replace("", pd.NA)

# Bỏ trùng hoàn toàn nếu Row ID có mặt; kiểm tra xung đột khóa dòng.
exact_duplicates = int(df.duplicated().sum())
if "Row ID" in df.columns:
    df = df.drop_duplicates().copy()
    row_id_conflicts = int(df["Row ID"].duplicated().sum())
    if row_id_conflicts:
        raise ValueError(
            f"Có {row_id_conflicts} Row ID lặp nhưng dữ liệu khác nhau. "
            "Cần kiểm tra thủ công trước khi tính KPI."
        )
    removed_duplicates = exact_duplicates
else:
    removed_duplicates = 0
    if exact_duplicates:
        print("CẢNH BÁO: thiếu Row ID, giữ nguyên các dòng trùng hoàn toàn để tránh xóa nhầm dòng hàng.")

# Chuyển kiểu số (hỗ trợ dấu $ và dấu phẩy phân cách hàng nghìn).
def parse_numeric(series):
    cleaned = (series.astype("string").str.strip()
               .str.replace("$", "", regex=False)
               .str.replace(",", "", regex=False))
    percent_flag = cleaned.str.contains("%", regex=False, na=False)
    cleaned = cleaned.str.replace("%", "", regex=False)
    result = pd.to_numeric(cleaned, errors="coerce")
    result.loc[percent_flag] = result.loc[percent_flag] / 100
    return result.astype(float)

for col in ["Sales", "Profit", "Discount", "Quantity"]:
    if col in df.columns:
        df[col] = parse_numeric(df[col])

# Ngày dạng MM/DD/YYYY; thử ISO khi giá trị chưa parse được.
def parse_superstore_date(series):
    text = series.astype("string").str.strip()
    parsed = pd.to_datetime(text, format="%m/%d/%Y", errors="coerce")
    unresolved = parsed.isna() & text.notna() & text.ne("")
    if unresolved.any():
        parsed.loc[unresolved] = pd.to_datetime(
            text.loc[unresolved], format="%Y-%m-%d", errors="coerce"
        )
    return parsed

for col in ["Order Date", "Ship Date"]:
    if col in df.columns:
        df[col] = parse_superstore_date(df[col])

# Cột bắt buộc cho KPI: không được thiếu.
invalid_core = df["Order ID"].isna() | df["Sales"].isna() | df["Profit"].isna()
removed_core = int(invalid_core.sum())
df = df.loc[~invalid_core].copy()
if df.empty:
    raise ValueError("Không còn dòng hợp lệ để phân tích. Kiểm tra cột Order ID/Sales/Profit.")

# Giữ bản ghi với thông tin chiều thiếu, nhưng ghi rõ Unknown.
for col in ["Category", "Sub-Category", "Region", "Segment", "Product Name"]:
    df[col] = df[col].fillna("Unknown")

quality = {
    "Dòng ban đầu": before_rows,
    "Trùng hoàn toàn đã loại": removed_duplicates,
    "Thiếu Order ID/Sales/Profit đã loại": removed_core,
    "Dòng còn lại": len(df),
    "Order Date không hợp lệ/thiếu": int(df["Order Date"].isna().sum()),
    "Sales <= 0 (chỉ cảnh báo)": int((df["Sales"] <= 0).sum()),
    "Profit < 0 (giữ nguyên)": int((df["Profit"] < 0).sum()),
    "Discount ngoài [0,1] hoặc thiếu": int((~df["Discount"].between(0, 1)).sum()),
}
if "Quantity" in df.columns:
    quality["Quantity <= 0 hoặc thiếu (cảnh báo)"] = int(
        (df["Quantity"].isna() | (df["Quantity"] <= 0)).sum()
    )
if "Ship Date" in df.columns:
    quality["Ship Date trước Order Date (cảnh báo)"] = int(
        (df["Ship Date"] < df["Order Date"]).sum()
    )
print("BÁO CÁO CHẤT LƯỢNG DỮ LIỆU")
display(pd.DataFrame(quality.items(), columns=["Kiểm tra", "Số dòng"]))

# Chỉ lấy các ngày hợp lệ khi phân tích theo thời gian.
time_df = df.dropna(subset=["Order Date"]).copy()
if time_df.empty:
    raise ValueError("Không parse được Order Date. Kiểm tra định dạng ngày trong CSV.")

# Xuất dữ liệu sạch để có thể dùng lại trong Power BI.
clean_path = OUTPUT_DIR / "superstore_clean.csv"
df.to_csv(clean_path, index=False, encoding="utf-8-sig", date_format="%Y-%m-%d")
print(f"Đã xuất: {clean_path}")

## 5. KPI tổng quan

Các KPI được tính trực tiếp trên `df` sau làm sạch. **Profit Margin = tổng Profit / tổng Sales**, không phải trung bình Profit Margin của từng dòng. `AOV` = tổng Sales / số `Order ID` duy nhất, không phải Sales trung bình trên một dòng hàng.

In [ ]:
def safe_divide(numerator, denominator):
    return numerator / denominator if pd.notna(denominator) and denominator != 0 else np.nan

total_sales = float(df["Sales"].sum())
total_profit = float(df["Profit"].sum())
total_orders = int(df["Order ID"].nunique())
profit_margin = 100 * safe_divide(total_profit, total_sales)
aov = safe_divide(total_sales, total_orders)

kpi = pd.DataFrame([
    ("Total Sales", f"${total_sales:,.2f}"),
    ("Total Profit", f"${total_profit:,.2f}"),
    ("Profit Margin", f"{profit_margin:.2f}%"),
    ("Total Orders", f"{total_orders:,}"),
    ("Average Order Value (AOV)", f"${aov:,.2f}"),
], columns=["KPI", "Giá trị tính từ CSV"])
display(kpi)
print("Ghi chú: Profit là trường lợi nhuận trong bộ dữ liệu; không tự coi là lợi nhuận ròng của doanh nghiệp.")

## 6. Phân tích theo thời gian

**Câu hỏi:** Tháng/năm nào có doanh số và lợi nhuận cao? Tất cả kết quả chỉ phản ánh các bản ghi có `Order Date` hợp lệ. Không diễn giải chênh lệch giữa các năm như tốc độ tăng trưởng bền vững nếu thời kỳ quan sát chưa đầy đủ.

In [ ]:
time_df["Month"] = time_df["Order Date"].dt.to_period("M").dt.to_timestamp()
time_df["Year"] = time_df["Order Date"].dt.year
monthly = (time_df.groupby("Month", as_index=False)
           .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
           .sort_values("Month"))
yearly = (time_df.groupby("Year", as_index=False)
          .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum")))

print("Doanh số/lợi nhuận theo năm:")
display(yearly.round(2))
print("5 tháng có doanh số cao nhất:")
display(monthly.nlargest(5, "Sales").round(2))

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
axes[0].plot(monthly["Month"], monthly["Sales"], marker="o", markersize=2)
axes[0].set(title="Monthly Sales Trend", ylabel="Sales (USD)")
axes[1].plot(monthly["Month"], monthly["Profit"], marker="o", markersize=2)
axes[1].axhline(0, linewidth=1, color="gray")
axes[1].set(title="Monthly Profit Trend", xlabel="Order Month", ylabel="Profit (USD)")
for ax in axes:
    ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_monthly_trends.png", dpi=160, bbox_inches="tight")
plt.show()

peak_month = monthly.loc[monthly["Sales"].idxmax()]
weak_month = monthly.loc[monthly["Profit"].idxmin()]
display(Markdown(
    f"**Nhận xét từ dữ liệu:** Tháng có Sales cao nhất là "
    f"**{peak_month['Month']:%m/%Y}** (${peak_month['Sales']:,.2f}). "
    f"Tháng có Profit thấp nhất là **{weak_month['Month']:%m/%Y}** "
    f"(${weak_month['Profit']:,.2f}). Chỉ số theo tháng chưa giải thích nguyên nhân biến động."
))

## 7. So sánh Category và Sub-Category

**Câu hỏi:** Nhóm hàng nào mang lại doanh thu nhưng lợi nhuận thấp? Tính Profit Margin từ **tổng** Sales và Profit của từng nhóm.

In [ ]:
def group_summary(col):
    result = (df.groupby(col, dropna=False)
              .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"),
                   OrderCount=("Order ID", "nunique"))
              .reset_index())
    result["Profit Margin (%)"] = np.where(
        result["Sales"].ne(0), result["Profit"] / result["Sales"] * 100, np.nan
    )
    return result.sort_values("Sales", ascending=False).reset_index(drop=True)

category = group_summary("Category")
subcategory = group_summary("Sub-Category")
print("Hiệu quả theo Category:")
display(category.round(2))
print("Sub-Category (sắp xếp theo Profit tăng dần):")
display(subcategory.sort_values("Profit").round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=category, x="Category", y="Sales", ax=axes[0])
sns.barplot(data=category, x="Category", y="Profit", ax=axes[1])
axes[0].set(title="Sales by Category", xlabel="Category", ylabel="Sales (USD)")
axes[1].set(title="Profit by Category", xlabel="Category", ylabel="Profit (USD)")
for ax in axes:
    ax.tick_params(axis="x", rotation=18)
    ax.yaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_category_performance.png", dpi=160, bbox_inches="tight")
plt.show()

best_cat = category.loc[category["Sales"].idxmax()]
lowest_sub = subcategory.loc[subcategory["Profit"].idxmin()]
display(Markdown(
    f"**Nhận xét từ dữ liệu:** Category có Sales lớn nhất là **{best_cat['Category']}** "
    f"(${best_cat['Sales']:,.2f}). Sub-Category có Profit thấp nhất là "
    f"**{lowest_sub['Sub-Category']}** (${lowest_sub['Profit']:,.2f}). "
    "Cần xem đồng thời Sales, Profit và biên lợi nhuận trước khi đánh giá hiệu quả."
))

## 8. So sánh Region và Segment

**Câu hỏi:** Khu vực và phân khúc khách hàng nào tạo doanh số tốt nhưng biên lợi nhuận yếu? `OrderCount` trong các nhóm là distinct trong từng nhóm, **không cộng dồn OrderCount các nhóm** nếu một đơn xuất hiện ở nhiều nhóm.

In [ ]:
region = group_summary("Region")
segment = group_summary("Segment")
print("Hiệu quả theo Region:")
display(region.round(2))
print("Hiệu quả theo Segment:")
display(segment.round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=region.sort_values("Profit Margin (%)"), x="Region",
            y="Profit Margin (%)", ax=axes[0])
sns.barplot(data=segment.sort_values("Profit Margin (%)"), x="Segment",
            y="Profit Margin (%)", ax=axes[1])
axes[0].set(title="Profit Margin by Region", xlabel="Region", ylabel="Profit Margin (%)")
axes[1].set(title="Profit Margin by Segment", xlabel="Segment", ylabel="Profit Margin (%)")
for ax in axes:
    ax.axhline(0, linewidth=1, color="gray")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_region_segment_margins.png", dpi=160, bbox_inches="tight")
plt.show()

weak_region = region.loc[region["Profit Margin (%)"].idxmin()]
best_segment = segment.loc[segment["Profit Margin (%)"].idxmax()]
display(Markdown(
    f"**Nhận xét từ dữ liệu:** Region có Profit Margin thấp nhất là "
    f"**{weak_region['Region']}** ({weak_region['Profit Margin (%)']:.2f}%). "
    f"Segment có Profit Margin cao nhất là **{best_segment['Segment']}** "
    f"({best_segment['Profit Margin (%)']:.2f}%). Đây là so sánh mô tả, chưa phải nguyên nhân."
))

## 9. Nhóm hàng và sản phẩm Sales cao nhưng Profit yếu

**Tiêu chí minh bạch:** Xét các `Sub-Category` có Sales từ phân vị 75% trở lên; gắn cờ nếu Profit âm **hoặc** Profit Margin thấp hơn mức chung của toàn bộ dữ liệu. Đây là **quy tắc sàng lọc**, không phải bằng chứng sản phẩm kinh doanh kém do giảm giá.

Đối với `Product Name`, dùng tổng Sales và Profit của tất cả dòng có cùng tên (nếu dữ liệu có `Product ID`, phân biệt theo cặp `Product ID`–`Product Name`).

In [ ]:
threshold = float(subcategory["Sales"].quantile(0.75))
high_sales_low_profit = subcategory.loc[
    (subcategory["Sales"] >= threshold) &
    ((subcategory["Profit"] < 0) |
     (subcategory["Profit Margin (%)"] < profit_margin))
].sort_values("Profit")
print(f"Ngưỡng Sub-Category có Sales cao (P75): ${threshold:,.2f}")
print("Các nhóm cần rà soát:")
display(high_sales_low_profit.round(2))

product_keys = ["Product ID", "Product Name"] if "Product ID" in df.columns else ["Product Name"]
products = (df.groupby(product_keys, dropna=False)
            .agg(Sales=("Sales", "sum"), Profit=("Profit", "sum"))
            .reset_index())
products["Profit Margin (%)"] = np.where(
    products["Sales"].ne(0), products["Profit"] / products["Sales"] * 100, np.nan
)
top_sales_products = products.nlargest(10, "Sales").copy()
loss_products = products[products["Profit"] < 0].sort_values("Profit")
print("10 sản phẩm có tổng Sales cao nhất:")
display(top_sales_products.round(2))
print(f"Số sản phẩm (theo khóa nhóm) có tổng Profit âm: {len(loss_products):,}")
display(loss_products.head(10).round(2))

top_sub = subcategory.nlargest(10, "Sales").sort_values("Sales")
fig, axes = plt.subplots(1, 2, figsize=(13, 7))
axes[0].barh(top_sub["Sub-Category"], top_sub["Sales"])
axes[0].set(title="Top 10 Sub-Categories by Sales", xlabel="Sales (USD)", ylabel="Sub-Category")
axes[1].barh(top_sub["Sub-Category"], top_sub["Profit"])
axes[1].axvline(0, linewidth=1, color="gray")
axes[1].set(title="Profit of Top-Sales Sub-Categories", xlabel="Profit (USD)", ylabel="Sub-Category")
for ax in axes:
    ax.xaxis.set_major_formatter(mtick.StrMethodFormatter('${x:,.0f}'))
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_subcategory_sales_profit.png", dpi=160, bbox_inches="tight")
plt.show()

if len(high_sales_low_profit):
    flagged_names = ", ".join(high_sales_low_profit["Sub-Category"].astype(str).head(5))
    note = f"Theo tiêu chí sàng lọc, các Sub-Category cần kiểm tra có: **{flagged_names}**."
else:
    note = "Không có Sub-Category nào thỏa đồng thời tiêu chí Sales cao và Profit yếu đã đặt."
display(Markdown("**Nhận xét từ dữ liệu:** " + note +
                 " Cần kiểm tra cấu trúc chi phí và chính sách giá trước khi kết luận nguyên nhân."))

## 10. Discount và Profit

**Câu hỏi:** Những dòng hàng có Discount cao có xu hướng Profit Margin thấp hơn không?

- `Discount` được hiểu là tỷ lệ từ 0 đến 1 (ví dụ 0.2 = 20%). Giá trị ngoài phạm vi không tham gia phân tích Discount.
- Đồ thị điểm thể hiện mối liên hệ ở cấp **dòng hàng**, có thể chọn ngẫu nhiên tối đa 3.000 dòng chỉ để dễ nhìn; bảng thống kê và hệ số tương quan sử dụng **toàn bộ** dòng hợp lệ.
- Tính Spearman correlation giữa Discount và Profit; hệ số này **không chứng minh tác động nhân quả**. Khi so sánh theo nhóm Discount phải tính margin = tổng Profit / tổng Sales, không lấy trung bình margin dòng.

In [ ]:
discount_df = df.loc[df["Discount"].between(0, 1)].copy()
if discount_df.empty:
    raise ValueError("Không có Discount hợp lệ trong [0,1] để phân tích.")

bins = [-0.000001, 0, 0.10, 0.20, 0.30, 1]
labels = ["0%", ">0–10%", ">10–20%", ">20–30%", ">30–100%"]
discount_df["Discount Group"] = pd.cut(
    discount_df["Discount"], bins=bins, labels=labels, include_lowest=True
)
by_discount = (discount_df.groupby("Discount Group", observed=True)
               .agg(Rows=("Profit", "size"), Sales=("Sales", "sum"),
                    Profit=("Profit", "sum"),
                    LossRows=("Profit", lambda x: int((x < 0).sum())))
               .reset_index())
by_discount["Profit Margin (%)"] = np.where(
    by_discount["Sales"].ne(0), by_discount["Profit"] / by_discount["Sales"] * 100, np.nan
)
by_discount["Loss rows (%)"] = 100 * by_discount["LossRows"] / by_discount["Rows"]
print("Bảng so sánh theo mức giảm giá (tất cả các dòng hợp lệ):")
display(by_discount.round(2))

rho = discount_df[["Discount", "Profit"]].corr(method="spearman").iloc[0, 1]
print(f"Spearman(Discount, Profit) = {rho:.3f}" if pd.notna(rho)
      else "Không đủ biến thiên dữ liệu để tính Spearman correlation.")

plot_df = discount_df.sample(n=min(len(discount_df), 3000), random_state=42)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=plot_df, x="Discount", y="Profit", alpha=0.35,
                s=25, ax=axes[0], legend=False)
axes[0].set(title="Discount vs. Profit (Line Items)", xlabel="Discount (%)", ylabel="Profit (USD)")
axes[0].xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
axes[0].axhline(0, linewidth=1, color="gray")
sns.barplot(data=by_discount, x="Discount Group", y="Profit Margin (%)", ax=axes[1])
axes[1].set(title="Profit Margin by Discount Band", xlabel="Discount Band", ylabel="Profit Margin (%)")
axes[1].tick_params(axis="x", rotation=20)
axes[1].axhline(0, linewidth=1, color="gray")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_discount_profit.png", dpi=160, bbox_inches="tight")
plt.show()

if pd.notna(rho):
    if rho < 0:
        tendency = "âm"
    elif rho > 0:
        tendency = "dương"
    else:
        tendency = "bằng 0"
    note = f"Spearman correlation giữa Discount và Profit là **{rho:.3f}** (dấu **{tendency}**)."
else:
    note = "Dữ liệu không đủ biến thiên để tính hệ số tương quan Spearman."
display(Markdown("**Nhận xét từ dữ liệu:** " + note +
                 " Đánh giá mức giảm giá cần xét thêm Category, Region và các yếu tố khác; không suy ra quan hệ nhân quả."))

## 11. Tổng hợp insight và đề xuất kinh doanh

Các kết luận bên dưới **được tạo tự động từ số liệu vừa tính**. Đề xuất là **hướng kiểm tra/ra quyết định có điều kiện**, không phải lời khẳng định rằng một thay đổi chính sách sẽ làm lợi nhuận tăng bao nhiêu.

In [ ]:
# Xây dựng insight có tham số số liệu thật, không điền tay kết quả.
insights = [
    f"**Quy mô:** Sales = ${total_sales:,.2f}; Profit = ${total_profit:,.2f}; "
    f"Profit Margin = {profit_margin:.2f}%; Orders = {total_orders:,}; AOV = ${aov:,.2f}.",
    f"**Thời gian:** Tháng doanh số cao nhất là {peak_month['Month']:%m/%Y} "
    f"với ${peak_month['Sales']:,.2f}; tháng lợi nhuận thấp nhất là "
    f"{weak_month['Month']:%m/%Y} với ${weak_month['Profit']:,.2f}.",
    f"**Danh mục:** {best_cat['Category']} có Sales cao nhất "
    f"(${best_cat['Sales']:,.2f}); {lowest_sub['Sub-Category']} "
    f"có Profit thấp nhất (${lowest_sub['Profit']:,.2f}).",
    f"**Khu vực/phân khúc:** {weak_region['Region']} có biên lợi nhuận thấp nhất "
    f"({weak_region['Profit Margin (%)']:.2f}%); {best_segment['Segment']} "
    f"có biên lợi nhuận cao nhất ({best_segment['Profit Margin (%)']:.2f}%).",
    f"**Sản phẩm:** Có {len(loss_products):,} khóa sản phẩm có tổng Profit âm; "
    f"có {len(high_sales_low_profit):,} Sub-Category được gắn cờ Sales cao / Profit yếu "
    "theo tiêu chí P75 và biên lợi nhuận chung.",
]
if pd.notna(rho):
    insights.append(
        f"**Giảm giá:** Spearman(Discount, Profit) = {rho:.3f} "
        "ở cấp dòng hàng. Đây là tương quan, **không phải bằng chứng nhân quả**."
    )
else:
    insights.append("**Giảm giá:** Không tính được tương quan Spearman từ dữ liệu hợp lệ.")

recommendations = [
    f"**Ưu tiên kiểm tra {weak_region['Region']}:** đối chiếu Discount, "
    "Sub-Category và chi phí để làm rõ nguyên nhân biên lợi nhuận thấp.",
]
if len(loss_products):
    recommendations.append(
        f"**Rà soát {len(loss_products):,} khóa sản phẩm có Profit âm:** "
        "kiểm tra giá bán, phí vận chuyển/hoàn trả và chi phí trước khi điều chỉnh chính sách."
    )
if len(high_sales_low_profit):
    recommendations.append(
        "**Theo dõi nhóm Sales cao nhưng Profit yếu:** ưu tiên phân tích chi tiết "
        "các Sub-Category đã được gắn cờ, thay vì tối ưu doanh thu bằng mọi giá."
    )
if pd.notna(rho):
    recommendations.append(
        "**Kiểm chứng chính sách giảm giá:** so sánh các mức Discount trong cùng "
        "Category/Region và cùng giai đoạn; không đề xuất mức Discount tối ưu chỉ từ tương quan."
    )
recommendations.append(
    "**Theo dõi hằng tháng:** Sales, Profit, Margin, Orders và AOV để phát hiện biến động mới."
)

report = (
    "## Kết quả chính (tính từ CSV)\n" +
    "\n".join(f"{i}. {x}" for i, x in enumerate(insights, 1)) +
    "\n\n## Hướng hành động cần xác minh thêm\n" +
    "\n".join(f"{i}. {x}" for i, x in enumerate(recommendations, 1)) +
    "\n\n**Giới hạn:** Đây là dữ liệu minh họa; phân tích mô tả và tương quan "
    "không đủ để xác nhận quan hệ nhân quả hoặc ước lượng tác động của chính sách. "
    "Doanh số trong dữ liệu không đồng nghĩa dòng tiền thực thu, và Profit chưa "
    "được xác minh là lợi nhuận ròng sau mọi chi phí."
)
display(Markdown(report))

# Lưu văn bản insight để tái sử dụng trong README/portfolio sau khi đã chạy dữ liệu thật.
report_path = OUTPUT_DIR / "analysis_insights.md"
report_path.write_text(report, encoding="utf-8")
print(f"Đã lưu kết quả phân tích: {report_path}")

## 12. Kết luận và bước phát triển tiếp theo

Notebook đã thiết lập quy trình có thể tái chạy: tải dữ liệu thật, kiểm tra chất lượng, xử lý theo quy tắc rõ ràng, tính KPI, so sánh theo thời gian/sản phẩm/khu vực/phân khúc, trực quan hóa và tạo kết luận tự động. **Kết quả định lượng cuối cùng chính là phần được sinh ở mục 11 sau khi chạy dữ liệu CSV đã tải lên**, không phải các con số chép sẵn trong Markdown.

**Bước tiếp theo dành cho CV Data Analyst Intern:**

1. Kiểm tra output và đảm bảo từng nhận xét khớp bảng/biểu đồ.
2. Tạo Power BI dashboard với 5 KPI, Monthly Sales/Profit, Margin theo Region và Sub-Category, bộ lọc Category/Segment/Year.
3. Chỉ đưa **số liệu đã kiểm chứng** và ảnh thật từ notebook/dashboard vào GitHub README và CV; không ghi thành tích kiểu “tăng lợi nhuận X%” khi chưa có triển khai thực tế.

**Các file tạo ra trên Colab:** `superstore_outputs/superstore_clean.csv`, `superstore_outputs/analysis_insights.md`, và 5 ảnh `.png`. Cell tiếp theo tải toàn bộ về máy.

In [ ]:
from google.colab import files
import shutil

archive = shutil.make_archive("superstore_outputs", "zip", root_dir=OUTPUT_DIR)
print("Đã đóng gói kết quả:", archive)
files.download(archive)